# Mini-Projeto 1 - Análise de Campanhas de Marketing com Power BI

## Importando as bibliotecas

In [89]:
#install
#!pip install -q -U duckdb matplotlib nbformat numpy pandas plotly scikit-learn scipy seaborn shap statsmodels xgboost

In [90]:
#imports
import pandas as pd
import numpy as np
import os
import duckdb as ddb
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import plotly.graph_objects as go

## Acessando os dados

In [91]:
dataset_capitulo_04 = pd.read_csv('dados_marketing/dados_marketing_clear.csv')

In [92]:
dataset_capitulo_04.head(5)

,ID,Ano Nascimento,Escolaridade,Estado Civil,Salario Anual,Filhos em Casa,Adolescentes em Casa,Data Cadastro,Dias Desde Ultima Compra,Gasto com Eletronicos,...,Numero de Compras via Catalogo,Numero de Compras na Loja,Numero Visitas WebSite Mes,Compra na Campanha 1,Compra na Campanha 2,Compra na Campanha 3,Compra na Campanha 4,Compra na Campanha 5,Comprou,Pais
0,2795,1958,Mestrado,Solteiro,30523.0,2,1,07/01/2020,0,5,...,0,2,7,0,0,0,0,0,0,Chile
1,2285,1954,Mestrado,Casado,36634.0,0,1,05/12/2023,0,213,...,2,5,7,0,0,0,0,0,0,Espanha
2,115,1966,Mestrado,Solteiro,43456.0,0,1,03/02/2023,0,275,...,1,8,5,0,0,0,0,0,0,Argentina
3,10470,1979,Mestrado,Solteiro,40662.0,1,0,03/05/2023,0,40,...,1,3,4,0,0,0,0,0,0,Alemanha
4,4065,1976,Doutorado,Solteiro,49544.0,1,0,02/11/2020,0,308,...,1,8,7,0,0,0,0,0,0,Estados Unidos


In [93]:
dataset_capitulo_04.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 27 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   ID                              2000 non-null   int64  
 1   Ano Nascimento                  2000 non-null   int64  
 2   Escolaridade                    2000 non-null   str    
 3   Estado Civil                    2000 non-null   str    
 4   Salario Anual                   2000 non-null   float64
 5   Filhos em Casa                  2000 non-null   int64  
 6   Adolescentes em Casa            2000 non-null   int64  
 7   Data Cadastro                   2000 non-null   str    
 8   Dias Desde Ultima Compra        2000 non-null   int64  
 9   Gasto com Eletronicos           2000 non-null   int64  
 10  Gasto com Brinquedos            2000 non-null   int64  
 11  Gasto com Moveis                2000 non-null   int64  
 12  Gasto com Utilidades            2000 non-null

## Visão do Cliente

### Qual é a quantidade total de usuários?

In [94]:
contagem_de_usuarios = dataset_capitulo_04['ID'].count()
contagem_de_usuarios

np.int64(2000)

In [95]:
print(f'Tem {contagem_de_usuarios} usuarios cadastrados')

Tem 2000 usuarios cadastrados


### Qual é o salário médio anual dos usuários?

In [96]:
salario_medio_anual = dataset_capitulo_04['Salario Anual'].mean()
salario_medio_anual

np.float64(51983.665)

In [97]:
print(f'O Salario médio anual dos clientes é de{salario_medio_anual: .2f} dinheiros.')

O Salario médio anual dos clientes é de 51983.67 dinheiros.


### Qual é a quantidade de compras realizadas em lojas físicas?

In [98]:
total_de_compras_em_lojas_fisicas = dataset_capitulo_04['Numero de Compras na Loja'].sum()
total_de_compras_em_lojas_fisicas

np.int64(11595)

In [99]:
print(f'A quantidade de compras feitas em lojas físicas é {total_de_compras_em_lojas_fisicas}.')

A quantidade de compras feitas em lojas físicas é 11595.


### Qual é a quantidade de compras realizadas na web?

In [100]:
total_de_compras_na_web = dataset_capitulo_04['Numero de Compras na Web'].sum()
print(f'A quantidade de compras feitas na Web é {total_de_compras_na_web}.')

A quantidade de compras feitas na Web é 8150.


### Qual é a quantidade de compras realizadas por catálogo?

In [101]:
total_de_compras_por_catálogo = dataset_capitulo_04['Numero de Compras via Catalogo'].sum()
print(f'A quantidade de compras feitas por catálogo é {total_de_compras_por_catálogo}.')

A quantidade de compras feitas por catálogo é 5271.


### Qual é a quantidade e o percentual de compras realizadas com desconto?

In [102]:
total_de_compras_com_desconto = dataset_capitulo_04['Numero de Compras com Desconto'].sum()
print(f'A quantidade de compras feitas com desconto é {total_de_compras_com_desconto}.')

A quantidade de compras feitas com desconto é 4665.


### Qual é a quantidade de usuários por nível de escolaridade?

In [103]:
usuarios_por_nivel_de_escoloridade = dataset_capitulo_04['Escolaridade'].value_counts().reset_index()
usuarios_por_nivel_de_escoloridade

,Escolaridade,count
0,Curso Superior,1000
1,Doutorado,437
2,Mestrado,337
3,Segundo Grau,177
4,Primeiro Grau,49


### Qual é a quantidade de usuários por estado civil?

In [104]:
usuarios_por_nivel_de_escoloridade = dataset_capitulo_04['Estado Civil'].value_counts().reset_index()
usuarios_por_nivel_de_escoloridade

,Estado Civil,count
0,Solteiro,1197
1,Casado,523
2,Divorciado,280


## Visão do Comportamento de Compra do Cliente

### Qual é o gasto total dos usuários por faixa de salário anual?

In [105]:
colunas = ['Gasto com Eletronicos','Gasto com Brinquedos','Gasto com Moveis','Gasto com Utilidades','Gasto com Alimentos','Gasto com Vestuario']
dataset_capitulo_04['Gasto Total'] = 0
for coluna in colunas:
    dataset_capitulo_04['Gasto Total'] += dataset_capitulo_04[coluna]


In [106]:
dataset_capitulo_04.loc[:5,['Gasto com Eletronicos','Gasto com Brinquedos','Gasto com Moveis','Gasto com Utilidades','Gasto com Alimentos','Gasto com Vestuario','Gasto Total']]

,Gasto com Eletronicos,Gasto com Brinquedos,Gasto com Moveis,Gasto com Utilidades,Gasto com Alimentos,Gasto com Vestuario,Gasto Total
0,5,0,3,0,0,5,13
1,213,9,76,4,3,30,335
2,275,11,68,25,7,7,393
3,40,2,23,0,4,23,92
4,308,0,73,0,0,23,404
5,266,21,300,65,8,44,704


In [107]:
gasto_total_por_usuario_em_relação_ao_salario_anual = dataset_capitulo_04.loc[:,['Salario Anual','Gasto Total']]
gasto_total_por_usuario_em_relação_ao_salario_anual.head(5)

,Salario Anual,Gasto Total
0,30523.0,13
1,36634.0,335
2,43456.0,393
3,40662.0,92
4,49544.0,404


### Qual é a quantidade de usuários por número de filhos?

In [108]:
quantidade_de_filhos_por_usuario = dataset_capitulo_04['Filhos em Casa'].value_counts()
quantidade_de_filhos_por_usuario

Filhos em Casa
0    1144
1     817
2      39
Name: count, dtype: int64

### Qual é a quantidade de usuários por número de adolescentes na família?

In [109]:
quantidade_de_adolocentes_em_casa_por_usuario = dataset_capitulo_04['Adolescentes em Casa'].value_counts()
quantidade_de_adolocentes_em_casa_por_usuario

Adolescentes em Casa
0    1036
1     922
2      42
Name: count, dtype: int64

## Visão da Performance das Campanhas de Marketing

### Qual é a quantidade de vendas por campanha?

In [110]:
colunas = ['Compra na Campanha 1','Compra na Campanha 2','Compra na Campanha 3','Compra na Campanha 4','Compra na Campanha 5']
for coluna in colunas:
    vendas_por_campanha = dataset_capitulo_04[coluna].value_counts()
    print(vendas_por_campanha)

Compra na Campanha 1
0    1852
1     148
Name: count, dtype: int64
Compra na Campanha 2
0    1854
1     146
Name: count, dtype: int64
Compra na Campanha 3
0    1857
1     143
Name: count, dtype: int64
Compra na Campanha 4
0    1867
1     133
Name: count, dtype: int64
Compra na Campanha 5
0    1974
1      26
Name: count, dtype: int64


### Qual é o salário médio anual dos usuários por campanha?

In [111]:
colunas = ['Compra na Campanha 1','Compra na Campanha 2','Compra na Campanha 3','Compra na Campanha 4','Compra na Campanha 5']
usuarios_por_campanha = []
for coluna in colunas:
    compradores_por_campanha = dataset_capitulo_04[coluna]
    usuarios_por_campanha.append({
        'Campanha':        coluna[10:],
        'Não Compradores': round(dataset_capitulo_04.loc[compradores_por_campanha == 0, 'Salario Anual'].mean(), 2),
        'Compradores':     round(dataset_capitulo_04.loc[compradores_por_campanha == 1, 'Salario Anual'].mean(), 2)
    })
salario_medio_de_usuarios_por_campanhas = pd.DataFrame(usuarios_por_campanha)
salario_medio_de_usuarios_por_campanhas

,Campanha,Não Compradores,Compradores
0,Campanha 1,52175.49,49583.26
1,Campanha 2,50677.56,68569.40
2,Campanha 3,49680.11,81897.72
3,Campanha 4,50089.63,78571.31
4,Campanha 5,51745.59,70059.35


### Qual é o perfil de compradores por estado civil e escolaridade em relação ao número de visitas no website?

In [112]:
escolaridade  = dataset_capitulo_04['Escolaridade'].unique()
estado_civil  = dataset_capitulo_04['Estado Civil'].unique()
compradores   = dataset_capitulo_04['Comprou'].unique()

linhas_do_perfil_de_compradores = []

for compra in compradores:
    for estado in estado_civil:
        for escola in escolaridade:
            filtro = (
                (dataset_capitulo_04['Comprou']      == compra) &
                (dataset_capitulo_04['Estado Civil'] == estado) &
                (dataset_capitulo_04['Escolaridade'] == escola)
            )
            total = dataset_capitulo_04.loc[filtro, 'Numero Visitas WebSite Mes'].sum()

            linhas_do_perfil_de_compradores.append({
                'Comprou':      compra,
                'Estado Civil': estado,
                'Escolaridade': escola,
                'Total Visitas': total
            })

dataframe_linhas_do_perfil_de_compradores = pd.DataFrame(linhas_do_perfil_de_compradores)

In [113]:
dataframe_pivot = dataframe_linhas_do_perfil_de_compradores.pivot_table(
    index   = ['Comprou', 'Estado Civil'],
    columns = 'Escolaridade',
    values  = 'Total Visitas',
    aggfunc = 'sum'
).fillna('-')

dataframe_pivot['Total geral'] = dataframe_linhas_do_perfil_de_compradores.groupby(
    ['Comprou', 'Estado Civil']
)['Total Visitas'].sum().values

print(dataframe_pivot)

Escolaridade          Curso Superior  Doutorado  Mestrado  Primeiro Grau  \
Comprou Estado Civil                                                       
0       Casado                  1178        502       427             71   
        Divorciado               618        244       174             13   
        Solteiro                2749       1074       852            236   
1       Casado                   161         80        78             16   
        Divorciado               104        129        60              0   
        Solteiro                 495        304       153              0   

Escolaridade          Segundo Grau  Total geral  
Comprou Estado Civil                             
0       Casado                 234         2412  
        Divorciado             129         1178  
        Solteiro               451         5362  
1       Casado                  17          352  
        Divorciado              19          312  
        Solteiro                87       

### Qual é a quantidade de usuários que compraram por número de filhos?

In [114]:
numero_de_filhos  = dataset_capitulo_04['Filhos em Casa'].unique()
compradores   = dataset_capitulo_04['Comprou'].unique()
numero_de_filhos.sort(0)
compradores_com_filhos = []

for compra in compradores:
    for filho in numero_de_filhos:
        filtro = (
            (dataset_capitulo_04['Comprou']      == compra) &
            (dataset_capitulo_04['Filhos em Casa'] == filho)
        )
        total = dataset_capitulo_04.loc[filtro, 'ID'].count()

        compradores_com_filhos.append({
            'Comprou':      compra,
            'Filhos em Casa': filho,
            'Total': total
        })

dataframe_compradores_com_filhos = pd.DataFrame(compradores_com_filhos)
dataframe_pivot = dataframe_compradores_com_filhos.pivot_table(
    index   = ['Comprou'],
    columns = 'Filhos em Casa',
    values  = 'Total',
    aggfunc = 'sum'
).fillna('-')

print(dataframe_pivot)

Filhos em Casa    0    1   2
Comprou                     
0               936  707  37
1               208  110   2


## Visão dos Padrões de Compra no Ponto de Venda (País)

### Qual é o total gasto por categoria de produto em relação aos países?

In [115]:
colunas = ['Gasto com Eletronicos','Gasto com Brinquedos','Gasto com Moveis','Gasto com Utilidades','Gasto com Alimentos','Gasto com Vestuario']
paises  = dataset_capitulo_04['Pais'].unique()
gasto_por_pais = []
for pais in paises:
    filtro = (dataset_capitulo_04['Pais'] == pais)
    linha = {'Pais': pais}
    for coluna in colunas:
        gasto = dataset_capitulo_04.loc[filtro, coluna].sum()
        linha[coluna[10:]] = round(gasto,2)
    linha['Contagem'] = dataset_capitulo_04.loc[dataset_capitulo_04['Pais'] == pais,'ID'].count()
    gasto_por_pais.append(linha)
pd.DataFrame(gasto_por_pais)

,Pais,Eletronicos,Brinquedos,Moveis,Utilidades,Alimentos,Vestuario,Contagem
0,Chile,76046,6916,40619,9035,6920,10854,245
1,Espanha,94291,7844,50491,11970,7825,13224,303
2,Argentina,33916,3635,21820,4271,2736,5675,133
3,Alemanha,30182,2671,16251,4218,2362,4884,104
4,Estados Unidos,304683,25491,160401,35977,27139,41283,977
5,Portugal,28823,2543,17771,4158,2535,4257,93
6,Brasil,39916,3533,20934,5545,4136,7090,145


### Qual é o total gasto por ano e por país?

In [120]:
dataset_capitulo_04['Data Cadastro'] = pd.to_datetime(dataset_capitulo_04['Data Cadastro'], format='%d/%m/%Y').dt.year
dataset_capitulo_04['Data Cadastro'].head()

0    2020
1    2023
2    2023
3    2023
4    2020
Name: Data Cadastro, dtype: int32

In [ ]:
anos   = dataset_capitulo_04['Data Cadastro'].unique()
paises = dataset_capitulo_04['Pais'].unique()

gasto_por_ano_por_pais = []

for pais in paises:
    for ano in anos:
        filtro = (
            (dataset_capitulo_04['Pais']         == pais) &
            (dataset_capitulo_04['Data Cadastro'] == ano)
        )
        gasto = dataset_capitulo_04.loc[filtro, 'Gasto Total'].sum()

        gasto_por_ano_por_pais.append({
            'Pais':         pais,
            'Ano':          ano,          # ← chave com o nome do ano
            'Gasto Total':  round(gasto, 2)
        })

df_gasto_por_ano_por_pais = pd.DataFrame(gasto_por_ano_por_pais)
df_gasto_por_ano_por_pais

,Pais,Ano,Gasto Total
0,Chile,2020,42213
1,Chile,2023,40164
2,Chile,2022,25953
3,Chile,2021,14719
4,Chile,2018,14761
5,Chile,2019,12580
6,Espanha,2020,37792
7,Espanha,2023,64006
8,Espanha,2022,33041
9,Espanha,2021,19539
